# PAD-Stream — Colab A100 Bootstrap

This notebook establishes the **compute worker** for PAD-Stream.

**Authority model**
- PyCharm / Windows: development authority
- Private GitHub repository: source/version authority
- Google Colab A100: CUDA compute worker
- Google Drive: persistent datasets, checkpoints, run outputs, environment manifests

This bootstrap does **not** start COMP-XPU-005 training and does **not** modify the certified XPU lineage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Persistent Drive layout
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/PAD_Stream_Compute')
DRIVE_DATASETS = DRIVE_ROOT / 'datasets'
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
DRIVE_RUNS = DRIVE_ROOT / 'runs'
DRIVE_PROVENANCE = DRIVE_ROOT / 'provenance'

for p in [DRIVE_ROOT, DRIVE_DATASETS, DRIVE_CHECKPOINTS, DRIVE_RUNS, DRIVE_PROVENANCE]:
    p.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
for p in [DRIVE_DATASETS, DRIVE_CHECKPOINTS, DRIVE_RUNS, DRIVE_PROVENANCE]:
    print(" ", p)

In [ ]:
# A100 / CUDA environment audit
import json, os, platform, subprocess, sys
from pathlib import Path
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Colab choose Runtime > Change runtime type > GPU.")

gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)
print("CUDA runtime:", torch.version.cuda)
print("CUDA device count:", torch.cuda.device_count())

if "A100" not in gpu_name.upper():
    raise RuntimeError(
        f"This PAD-Stream compute session is expected to use an A100, but Colab assigned: {gpu_name}"
    )

subprocess.run(["nvidia-smi"], check=False)

## Private GitHub access

Create a Colab Secret named **`GITHUB_TOKEN`** containing a GitHub personal access token that can read your private PAD-Stream repository.

Then set `REPO_SLUG` below to the final private repository name, for example:

`khb-git/PAD-Stream`

The token is read from Colab Secrets and is not written into the notebook.

In [ ]:
# EDIT THIS ONCE THE PRIVATE REPOSITORY EXISTS.
REPO_SLUG = "khb-git/PAD-Stream"

from google.colab import userdata
import os, subprocess
from pathlib import Path

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN is missing.")

WORK_ROOT = Path("/content")
REPO_DIR = WORK_ROOT / REPO_SLUG.split("/")[-1]

# Use an askpass helper so the PAT is not embedded in the repository remote URL.
askpass = Path("/content/padstream_git_askpass.sh")
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$PADSTREAM_GITHUB_TOKEN" ;;\n'
    'esac\n'
)
askpass.chmod(0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"
env["PADSTREAM_GITHUB_TOKEN"] = token

remote = f"https://github.com/{REPO_SLUG}.git"

if not (REPO_DIR / ".git").exists():
    print("Cloning", REPO_SLUG)
    proc = subprocess.run(
        ["git", "clone", remote, str(REPO_DIR)],
        env=env, text=True, capture_output=True
    )
else:
    print("Updating existing Colab working copy", REPO_DIR)
    proc = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        env=env, text=True, capture_output=True
    )

# Redact token defensively before showing errors.
stdout = (proc.stdout or "").replace(token, "***REDACTED***")
stderr = (proc.stderr or "").replace(token, "***REDACTED***")
print(stdout)
if proc.returncode != 0:
    print(stderr)
    raise RuntimeError("Git clone/pull failed. See redacted output above.")

print("Repository:", REPO_DIR)

In [ ]:
# Git / source provenance audit
import hashlib, json, subprocess
from pathlib import Path

def sha256(path: Path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

status = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
)

print("Git commit:", commit)
print("Working tree clean:", not bool(status.strip()))
if status.strip():
    print(status)

source_manifest = {}
for path in sorted(REPO_DIR.rglob("*.py")):
    if ".git" in path.parts or "__pycache__" in path.parts:
        continue
    source_manifest[str(path.relative_to(REPO_DIR))] = sha256(path)

manifest = {
    "git_commit": commit,
    "working_tree_clean": not bool(status.strip()),
    "python_source_sha256": source_manifest,
}

manifest_path = DRIVE_PROVENANCE / "PAD_Stream_Colab_Source_Manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Source manifest:", manifest_path)
print("Python source files hashed:", len(source_manifest))

In [ ]:
# Save CUDA/A100 environment manifest to Drive
import json, platform, subprocess, sys, torch
from datetime import datetime, timezone

env_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "git_commit": commit,
}

env_path = DRIVE_PROVENANCE / "PAD_Stream_Colab_A100_Environment.json"
env_path.write_text(json.dumps(env_manifest, indent=2), encoding="utf-8")
print(env_path)
print(json.dumps(env_manifest, indent=2))

## Next milestone

Once this bootstrap passes, do **not** launch the 800-epoch run yet.

The next PAD-Stream CUDA milestone should:

1. bind the Git commit + source SHA manifest;
2. verify the A100/CUDA backend;
3. reproduce the corrected derivative tensors and parameter gradients against the certified reference;
4. benchmark A100 cotangent chunks and microbatch sizes;
5. run one full production-shaped logical-batch CUDA rehearsal;
6. only then authorize the fresh CUDA COMP-XPU-005 production run.